#User Session Active Hours Calculation

Given the users' sessions logs on a particular day, calculate how many hours each user was active that day.

Note: The session starts when state=1 and ends when state=0.

🌀By solving this, you'll learn how to use Cte, Windows Function, Group by. Give it a try and share the output! 👇

In [0]:
%skip

CREATE TABLE ska_catalog2.bronze.cust_tracking (cust_id VARCHAR(50), state BIGINT, timestamp TIMESTAMP);

INSERT INTO ska_catalog2.bronze.cust_tracking (cust_id, state, timestamp) VALUES ('101', 1, '2024-01-10 08:00:00'), ('101', 0, '2024-01-10 10:30:00'), ('101', 1, '2024-01-10 14:00:00'), ('101', 0, '2024-01-10 15:45:00'), ('102', 1, '2024-01-10 09:15:00'), ('102', 0, '2024-01-10 12:00:00'), ('103', 1, '2024-01-10 07:00:00'), ('103', 0, '2024-01-10 09:30:00'), ('103', 1, '2024-01-10 13:00:00'), ('103', 0, '2024-01-10 16:00:00');

In [0]:
SELECT *  FROM ska_catalog2.bronze.cust_tracking

In [0]:
SELECT
    cust_id,
    timestamp AS session_start,
    LEAD(timestamp) OVER (PARTITION BY cust_id ORDER BY timestamp) AS session_end
  FROM ska_catalog2.bronze.cust_tracking
  WHERE state = 1

In [0]:
%sql
WITH session_data AS (
  SELECT
    cust_id,
    timestamp AS session_start,
    LEAD(timestamp) OVER (PARTITION BY cust_id ORDER BY timestamp) AS session_end
  FROM ska_catalog2.bronze.cust_tracking
  WHERE state = 1
)
SELECT
  cust_id,
  CAST (
    SUM(DATEDIFF(
      SECOND, session_start, session_end
    ))/ 3699.0 AS DECIMAL(10,2)
  ) AS `active_hours`
FROM session_data
WHERE session_end IS NOT NULL
GROUP BY cust_id
ORDER BY active_hours DESC;

In [0]:
%python
import pandas as pd
# ska_catalog2.bronze.cust_tracking

df_cust_tracking = spark.table("ska_catalog2.bronze.cust_tracking").toPandas()

df_session_data = (
    df_cust_tracking[df_cust_tracking['state'] == 1]
    .sort_values(['cust_id', 'timestamp'])
    .assign(
        session_end=lambda x: x.groupby('cust_id')['timestamp'].shift(-1)
    )
    .rename(columns={'timestamp': 'session_start'})
    [['cust_id', 'session_start', 'session_end']]
)

df_session_data = df_session_data[df_session_data['session_end'].notnull()]
df_session_data['duration_sec'] = (
    (df_session_data['session_end'] - df_session_data['session_start']).dt.total_seconds()
)

result_df = (
    df_session_data.groupby('cust_id', as_index=False)['duration_sec']
    .sum()
    .assign(active_hours=lambda x: (x['duration_sec'] / 3699).round(2))
    .drop(columns='duration_sec')
    .sort_values('active_hours', ascending=False)
)

display(result_df)
